# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:

gm_2007 = px.data.gapminder().query("year == 2007").copy()

global_mean = gm_2007['lifeExp'].mean()

gm_2007['lifeExp_deviation'] = gm_2007['lifeExp'] - global_mean

continent_deviation = (
    gm_2007.groupby('continent')['lifeExp_deviation']
    .mean()
    .sort_values()
)

lowest_continent = continent_deviation.index[0]
lowest_deviation = continent_deviation.iloc[0]

fig = px.choropleth(
    gm_2007,
    locations='iso_alpha',
    color='lifeExp_deviation',
    hover_name='country',
    hover_data={
        'lifeExp': ':.2f',
        'lifeExp_deviation': ':.2f',
        'iso_alpha': False
    },
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title=(
        f'{lowest_continent} Has the Lowest Average Life Expectancy '
        f'Relative to the World Average ({global_mean:.1f} Years)'
    ),
    labels={
        'lifeExp_deviation': 'Difference from global average (years)',
        'lifeExp': 'Life expectancy'
    }
)

fig.update_layout(
    template='plotly_white',
    coloraxis_colorbar_title='Years above/below<br>global average'
)

fig.show()

print(f"Global average life expectancy: {global_mean:.2f} years")
print(f"Continent furthest below average: {lowest_continent}")
print(f"Average difference: {lowest_deviation:.2f} years")


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** *(URL or description)*

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox`) **and reason:**

*Your answer here* : I chose px.choropleth because it is suitable for displaying country-level geographical data using GeoJSON boundaries. It allows us to compare GDP per capita across different countries on a world map.

**Colour scale chosen** (sequential or diverging) **and reason:**

*Your answer here* : I chose a sequential colour scale because GDP per capita is a numerical value that increases from lower to higher values. The Viridis colour scale makes it easier to identify countries with relatively low and high GDP per capita.


In [ ]:

import pandas as pd
import plotly.express as px
import json
from urllib.request import urlopen

geojson_url = "https://raw.githubusercontent.com/datasets/geo-countries/master/data/countries.geojson"

with urlopen(geojson_url) as response:
    my_geojson = json.load(response)

print(my_geojson['features'][0]['properties'])

gm_2007 = px.data.gapminder().query("year == 2007").copy()

gm_2007 = gm_2007.rename(
    columns={'iso_alpha': 'country_code'}
)

gm_2007['gdpPercap'] = pd.to_numeric(
    gm_2007['gdpPercap'],
    errors='coerce'
)

gm_2007 = gm_2007.dropna(
    subset=['country_code', 'gdpPercap']
)

highest_country = gm_2007.loc[
    gm_2007['gdpPercap'].idxmax(), 'country'
]

highest_gdp = gm_2007['gdpPercap'].max()

fig = px.choropleth(
    gm_2007,
    geojson=my_geojson,
    locations='country_code',
    featureidkey='properties.ISO_A3',
    color='gdpPercap',
    hover_name='country',
    hover_data={
        'gdpPercap': ':,.2f',
        'country_code': False
    },
    color_continuous_scale='Viridis',
    title=f'{highest_country} Has the Highest GDP per Capita in 2007',
    labels={
        'gdpPercap': 'GDP per capita (USD)'
    }
)

fig.update_geos(
    showframe=False,
    showcoastlines=True,
    projection_type='natural earth'
)

fig.update_layout(
    template='plotly_white',
    coloraxis_colorbar_title='GDP per capita<br>(USD)'
)

fig.show()

print("Country with highest GDP per capita:", highest_country)
print(f"Highest GDP per capita: ${highest_gdp:,.2f}")
